# Canonical (Information) Form of a Gaussian

So far, Gaussian distributions have been represented using the **moment form**:

$$
\mathcal{N}(\boldsymbol{\mu}, \Sigma)
$$

where

- $\boldsymbol{\mu}$ is the mean vector,
- $\Sigma$ is the covariance matrix.

This representation is intuitive because it directly describes the center and uncertainty of the Gaussian.

However, several Gaussian inference operations repeatedly involve the inverse covariance matrix:

$$
\Sigma^{-1}.
$$

This motivates a second representation: the **canonical form**, also called the **information form**.

In this notebook we will study:

- the precision matrix,
- the information vector,
- conversion between moment and canonical forms,
- why Gaussian multiplication becomes simpler in canonical form,
- implementation of a reusable `CanonicalGaussian` class,
- comparison between moment and canonical representations.


## 1. Motivation

Consider the product of two Gaussian beliefs.

In moment form,

$$
\Sigma=
\left(
\Sigma_1^{-1}
+
\Sigma_2^{-1}
\right)^{-1}
$$

and

$$
\boldsymbol{\mu}=
\Sigma
\left(
\Sigma_1^{-1}\boldsymbol{\mu}_1
+
\Sigma_2^{-1}\boldsymbol{\mu}_2
\right).
$$

The same inverse covariance terms appear repeatedly.

This suggests that for inference, it may be useful to represent the Gaussian directly using quantities involving

$$
\Sigma^{-1}.
$$


## 2. Representation Flowchart

```text
                 Gaussian Distribution
                         │
               ┌─────────┴─────────┐
               │                   │
               ▼                   ▼
          Moment Form         Canonical Form
            (μ, Σ)              (η, Λ)
               │                   │
               └─────────┬─────────┘
                         ▼
                  Same Gaussian
```

The two forms represent exactly the same probability distribution.

Only the parameterization changes.


## 3. Moment Form

The familiar representation is

$$
\mathbf{x}
\sim
\mathcal{N}(\boldsymbol{\mu}, \Sigma).
$$

Its probability density is

$$
p(\mathbf{x})=
\frac{1}
{\sqrt{(2\pi)^d|\Sigma|}}
\exp
\left(
-\frac{1}{2}
(\mathbf{x}-\boldsymbol{\mu})^\top
\Sigma^{-1}
(\mathbf{x}-\boldsymbol{\mu})
\right).
$$

The mean tells us where the distribution is centered.

The covariance tells us how uncertainty is distributed around that center.


## 4. Precision Matrix

Define

$$
\boxed{
\Lambda
=
\Sigma^{-1}
}
$$

where $\Lambda$ is called the **precision matrix** or **information matrix**.

The intuition is simple:

```text
Large covariance
      ↓
High uncertainty
      ↓
Low precision
```

and

```text
Small covariance
      ↓
Low uncertainty
      ↓
High precision
```

So covariance measures uncertainty, while precision measures confidence.


## 5. Information Vector

The second canonical parameter is

$$
\boxed{
\boldsymbol{\eta}
=
\Lambda\boldsymbol{\mu}
}
$$

where $\boldsymbol{\eta}$ is called the **information vector**.

A useful intuition is:

- $\boldsymbol{\mu}$ tells us **where** the belief is,
- $\Lambda$ tells us **how confident** the belief is,
- $\boldsymbol{\eta}=\Lambda\boldsymbol{\mu}$ represents the location weighted by confidence.

This is exactly the same quantity that appeared when multiplying Gaussian beliefs.


## 6. Canonical Form of the Gaussian

Using

$$
\Lambda=\Sigma^{-1}
$$

and

$$
\boldsymbol{\eta}=\Lambda\boldsymbol{\mu},
$$

the Gaussian density can be rewritten in canonical form as

$$
p(\mathbf{x})
\propto
\exp
\left(
-\frac{1}{2}
\mathbf{x}^\top
\Lambda
\mathbf{x}
+
\boldsymbol{\eta}^\top
\mathbf{x}
\right).
$$

The normalization constant is omitted here because many inference operations only need proportionality.

The important point is that the quadratic and linear terms are controlled directly by

$$
\Lambda
$$

and

$$
\boldsymbol{\eta}.
$$


## 7. Converting Moment Form to Canonical Form

Starting from

$$
(\boldsymbol{\mu},\Sigma),
$$

compute

$$
\boxed{
\Lambda=
\Sigma^{-1}
}
$$

and then

$$
\boxed{
\boldsymbol{\eta}=
\Lambda\boldsymbol{\mu}.
}
$$

Flow:

```text
Moment Form
(μ, Σ)
   │
   ├── invert Σ
   ▼
Λ = Σ⁻¹
   │
   ├── multiply by μ
   ▼
η = Λ μ
   │
   ▼
Canonical Form
(η, Λ)
```


In [1]:
import numpy as np

mean = np.array([2.0, 3.0])

covariance = np.array([
    [2.0, 0.5],
    [0.5, 1.0],
])

precision = np.linalg.inv(covariance)

information = precision @ mean

print("Precision matrix:")
print(precision)

print("\nInformation vector:")
print(information)


Precision matrix:
[[ 0.57142857 -0.28571429]
 [-0.28571429  1.14285714]]

Information vector:
[0.28571429 2.85714286]


## 8. Converting Canonical Form Back to Moment Form

Starting from

$$
(\boldsymbol{\eta},\Lambda),
$$

recover the covariance:

$$
\boxed{
\Sigma=
\Lambda^{-1}
}
$$

and then recover the mean:

$$
\boxed{
\boldsymbol{\mu}=
\Sigma\boldsymbol{\eta}.
}
$$

Flow:

```text
Canonical Form
(η, Λ)
   │
   ├── invert Λ
   ▼
Σ = Λ⁻¹
   │
   ├── multiply by η
   ▼
μ = Σ η
   │
   ▼
Moment Form
(μ, Σ)
```


In [2]:
recovered_covariance = np.linalg.inv(
    precision
)

recovered_mean = (
    recovered_covariance
    @ information
)

print("Recovered mean:")
print(recovered_mean)

print("\nRecovered covariance:")
print(recovered_covariance)


Recovered mean:
[2. 3.]

Recovered covariance:
[[2.  0.5]
 [0.5 1. ]]


## 9. Why Canonical Form Is Useful

The biggest advantage appears when multiplying Gaussian distributions.

Suppose

$$
p_1(\mathbf{x})
\propto
\exp
\left(
-\frac{1}{2}
\mathbf{x}^\top
\Lambda_1
\mathbf{x}
+
\boldsymbol{\eta}_1^\top
\mathbf{x}
\right)
$$

and

$$
p_2(\mathbf{x})
\propto
\exp
\left(
-\frac{1}{2}
\mathbf{x}^\top
\Lambda_2
\mathbf{x}
+
\boldsymbol{\eta}_2^\top
\mathbf{x}
\right).
$$

Multiplication adds the exponents:

$$
p_1(\mathbf{x})p_2(\mathbf{x})
\propto
\exp
\left(
-\frac{1}{2}
\mathbf{x}^\top
(\Lambda_1+\Lambda_2)
\mathbf{x}
+
(\boldsymbol{\eta}_1+\boldsymbol{\eta}_2)^\top
\mathbf{x}
\right).
$$

Therefore,

$$
\boxed{
\Lambda=
\Lambda_1+\Lambda_2
}
$$

and

$$
\boxed{
\boldsymbol{\eta}=
\boldsymbol{\eta}_1+\boldsymbol{\eta}_2.
}
$$

This is much simpler than repeatedly working with covariance inverses.


## 10. Multiplication Flowchart

```text
Gaussian 1                 Gaussian 2
(η₁, Λ₁)                   (η₂, Λ₂)
    │                          │
    └────────────┬─────────────┘
                 ▼
          Add information
                 │
        Λ = Λ₁ + Λ₂
        η = η₁ + η₂
                 │
                 ▼
          Combined Gaussian
```

The interpretation is also natural:

> Independent information adds.


In [3]:
mean_1 = np.array([0.0])
covariance_1 = np.array([[4.0]])

mean_2 = np.array([3.0])
covariance_2 = np.array([[1.0]])

precision_1 = np.linalg.inv(
    covariance_1
)
precision_2 = np.linalg.inv(
    covariance_2
)

information_1 = (
    precision_1 @ mean_1
)
information_2 = (
    precision_2 @ mean_2
)

combined_precision = (
    precision_1 + precision_2
)

combined_information = (
    information_1 + information_2
)

combined_covariance = np.linalg.inv(
    combined_precision
)

combined_mean = (
    combined_covariance
    @ combined_information
)

print("Combined mean:")
print(combined_mean)

print("\nCombined covariance:")
print(combined_covariance)


Combined mean:
[2.4]

Combined covariance:
[[0.8]]


## 11. Standalone Conversion Functions

Before creating the final class, implement the basic conversions separately.


In [4]:
def moment_to_canonical(
    mean: np.ndarray,
    covariance: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    mean = np.asarray(
        mean,
        dtype=float,
    )

    covariance = np.asarray(
        covariance,
        dtype=float,
    )

    precision = np.linalg.inv(
        covariance
    )

    information = (
        precision @ mean
    )

    return information, precision


In [5]:
def canonical_to_moment(
    information: np.ndarray,
    precision: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    information = np.asarray(
        information,
        dtype=float,
    )

    precision = np.asarray(
        precision,
        dtype=float,
    )

    covariance = np.linalg.inv(
        precision
    )

    mean = covariance @ information

    return mean, covariance


## 12. Verification

Convert a Gaussian from moment form to canonical form and back again.


In [7]:
information, precision = moment_to_canonical(
    mean,
    covariance,
)

recovered_mean, recovered_covariance = canonical_to_moment(
    information,
    precision,
)

print("Original mean:")
print(mean)

print("\nRecovered mean:")
print(recovered_mean)

print("\nOriginal covariance:")
print(covariance)

print("\nRecovered covariance:")
print(recovered_covariance)


Original mean:
[2. 3.]

Recovered mean:
[2. 3.]

Original covariance:
[[2.  0.5]
 [0.5 1. ]]

Recovered covariance:
[[2.  0.5]
 [0.5 1. ]]


## 13. Final Class

Now combine the canonical representation into one reusable object.

This class stores

$$
(\boldsymbol{\eta},\Lambda)
$$

rather than

$$
(\boldsymbol{\mu},\Sigma).
$$

It can:

- convert from moment form,
- convert back to moment form,
- multiply Gaussian information efficiently.


In [9]:
class CanonicalGaussian:
    """Gaussian distribution in canonical / information form."""

    def __init__(
        self,
        information: np.ndarray,
        precision: np.ndarray,
    ) -> None:
        self.information = np.asarray(
            information,
            dtype=float,
        )

        self.precision = np.asarray(
            precision,
            dtype=float,
        )

        self._validate_parameters()

    @property
    def dimension(self) -> int:
        return self.information.size

    def _validate_parameters(self) -> None:
        if self.information.ndim != 1:
            raise ValueError(
                "Information vector must be one-dimensional."
            )

        if self.precision.shape != (
            self.dimension,
            self.dimension,
        ):
            raise ValueError(
                "Precision matrix has incompatible shape."
            )

        if not np.allclose(
            self.precision,
            self.precision.T,
        ):
            raise ValueError(
                "Precision matrix must be symmetric."
            )

        eigenvalues = np.linalg.eigvalsh(
            self.precision
        )

        if np.any(eigenvalues <= 0):
            raise ValueError(
                "Precision matrix must be positive definite."
            )

    @classmethod
    def from_moment(
        cls,
        mean: np.ndarray,
        covariance: np.ndarray,
    ) -> "CanonicalGaussian":
        mean = np.asarray(
            mean,
            dtype=float,
        )

        covariance = np.asarray(
            covariance,
            dtype=float,
        )

        precision = np.linalg.inv(
            covariance
        )

        information = (
            precision @ mean
        )

        return cls(
            information=information,
            precision=precision,
        )

    def to_moment(
        self,
    ) -> tuple[np.ndarray, np.ndarray]:
        covariance = np.linalg.inv(
            self.precision
        )

        mean = (
            covariance
            @ self.information
        )

        return mean, covariance

    def multiply(
        self,
        other: "CanonicalGaussian",
    ) -> "CanonicalGaussian":
        if not isinstance(
            other,
            CanonicalGaussian,
        ):
            raise TypeError(
                "other must be a CanonicalGaussian."
            )

        if self.dimension != other.dimension:
            raise ValueError(
                "Both Gaussians must have the same dimension."
            )

        combined_information = (
            self.information
            + other.information
        )

        combined_precision = (
            self.precision
            + other.precision
        )

        return CanonicalGaussian(
            information=combined_information,
            precision=combined_precision,
        )


## 14. Class Example


In [11]:
canonical_1 = CanonicalGaussian.from_moment(
    mean=np.array([0.0]),
    covariance=np.array([[4.0]]),
)

canonical_2 = CanonicalGaussian.from_moment(
    mean=np.array([3.0]),
    covariance=np.array([[1.0]]),
)

combined = canonical_1.multiply(
    canonical_2
)

combined_mean, combined_covariance = (
    combined.to_moment()
)

print("Combined mean:")
print(combined_mean)

print("\nCombined covariance:")
print(combined_covariance)


Combined mean:
[2.4]

Combined covariance:
[[0.8]]


## 15. Moment Form vs Canonical Form

| Property / Operation | Moment Form $(\mu,\Sigma)$ | Canonical Form $(\eta,\Lambda)$ |
|---|---|---|
| Intuition | Very intuitive | Less intuitive |
| Center of distribution | Explicitly stored as $\mu$ | Recovered from $\Lambda^{-1}\eta$ |
| Uncertainty | Explicitly stored as $\Sigma$ | Represented through precision $\Lambda$ |
| Precision | Requires $\Sigma^{-1}$ | Stored directly |
| Sampling | Natural | Usually convert to moment form |
| Visualization | Natural | Usually convert to moment form |
| Marginalization | Simple | More involved |
| Conditioning | Convenient | Often efficient |
| Gaussian product | Requires covariance algebra | Very simple: add \(\eta\) and $\Lambda$ |
| Combining independent information | Less direct | Natural |
| Sparse graphical-model inference | Less convenient | Often preferred |
| Gaussian message passing | Possible | Particularly convenient |

The two representations are complementary.

Moment form is usually better for understanding, sampling, and visualization.

Canonical form is often better when accumulating and combining information during inference.


## 16. Final Conceptual Flow

```text
Gaussian belief
     │
     ├──────────────────────────────┐
     │                              │
     ▼                              ▼
Moment Form                    Canonical Form
(μ, Σ)                         (η, Λ)
     │                              │
     │ intuitive                    │ inference-friendly
     │                              │
     │                              ├── multiplication → addition
     │                              ├── information accumulation
     │                              └── sparse message passing
     │                              │
     └──────────────┬───────────────┘
                    ▼
             Same Gaussian
```

The important lesson is not that one representation is universally better.

Instead:

> Choose the representation that makes the current operation easier.
